In [1]:
import os
import copy
import numpy as np
import wfdb
import scipy.signal as signal
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [2]:
def bandpass_filter(signal_data, fs=100, lowcut=0.5, highcut=40, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')
    return signal.filtfilt(b, a, signal_data)

def notch_filter(signal_data, fs=100, freq=50.0, Q=30.0):
    nyq = 0.5 * fs
    w0 = freq / nyq
    b, a = signal.iirnotch(w0, Q)
    return signal.filtfilt(b, a, signal_data)

def preprocess_signal(sig, fs=100):
    sig = np.asarray(sig, dtype=np.float32)
    sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)

    sig = bandpass_filter(sig, fs)
    sig = notch_filter(sig, fs)

    sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)
    std = np.std(sig)
    if std < 1e-8:
        return np.zeros_like(sig, dtype=np.float32)

    sig = (sig - np.mean(sig)) / (std + 1e-8)
    return sig.astype(np.float32)


In [3]:
def segment_signal(signal_data, annotations, fs=100):
    segments = []
    labels = []
    segment_length = fs * 60

    for i, label in enumerate(annotations):
        start = i * segment_length
        end = start + segment_length

        if end <= len(signal_data):
            seg = signal_data[start:end]
            if not np.isfinite(seg).all():
                continue
            segments.append(seg)
            labels.append(1 if label == 'A' else 0)

    return np.array(segments, dtype=np.float32), np.array(labels, dtype=np.int64)


In [4]:
def load_apnea_dataset(data_path):
    X_all = []
    y_all = []
    skipped = []

    # use records that have apnea annotations
    records = sorted(set(f.split('.')[0] for f in os.listdir(data_path) if f.endswith('.apn')))

    for idx, rec in enumerate(records, 1):
        try:
            record = wfdb.rdrecord(os.path.join(data_path, rec))
            annotation = wfdb.rdann(os.path.join(data_path, rec), 'apn')

            signal_data = record.p_signal[:, 0]
            signal_data = preprocess_signal(signal_data, fs=int(record.fs))

            segments, labels = segment_signal(signal_data, annotation.symbol, fs=int(record.fs))
            if len(segments) == 0:
                skipped.append((rec, 'no valid segments'))
                continue

            X_all.append(segments)
            y_all.append(labels)

            if idx % 10 == 0:
                print(f"Loaded {idx}/{len(records)} records")

        except Exception as e:
            skipped.append((rec, str(e)))

    if not X_all:
        raise ValueError('No valid records were loaded.')

    X_all = np.concatenate(X_all, axis=0).astype(np.float32)
    y_all = np.concatenate(y_all, axis=0).astype(np.int64)

    finite_ratio = float(np.isfinite(X_all).mean())
    class_counts = np.bincount(y_all, minlength=2)
    print(f"Total segments: {len(X_all)} | finite_ratio={finite_ratio:.6f}")
    print(f"Class counts -> Normal(0): {class_counts[0]}, Apnea(1): {class_counts[1]}")
    if skipped:
        print(f"Skipped records: {len(skipped)}")

    return X_all, y_all


In [5]:
data_path = "dataset/1.0.0"

X, y = load_apnea_dataset(data_path)

# 80/10/10 split: train/val/test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f"Train samples: {len(X_train)}, Val samples: {len(X_val)}, Test samples: {len(X_test)}")
train_counts = np.bincount(y_train, minlength=2)
val_counts = np.bincount(y_val, minlength=2)
test_counts = np.bincount(y_test, minlength=2)
print(f"Train class counts -> 0: {train_counts[0]}, 1: {train_counts[1]}")
print(f"Val class counts   -> 0: {val_counts[0]}, 1: {val_counts[1]}")
print(f"Test class counts  -> 0: {test_counts[0]}, 1: {test_counts[1]}")


Loaded 10/86 records
Loaded 20/86 records
Loaded 50/86 records
Loaded 60/86 records
Loaded 70/86 records
Loaded 80/86 records
Total segments: 38222 | finite_ratio=1.000000
Class counts -> Normal(0): 23555, Apnea(1): 14667
Skipped records: 8
Train samples: 30577, Val samples: 3822, Test samples: 3823
Train class counts -> 0: 18844, 1: 11733
Val class counts   -> 0: 2355, 1: 1467
Test class counts  -> 0: 2356, 1: 1467


In [6]:
class ApneaDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(ApneaDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(ApneaDataset(X_val, y_val), batch_size=32)
test_loader = DataLoader(ApneaDataset(X_test, y_test), batch_size=32)


In [7]:
# --- 1. PCSA Block Optimized for Apnea ---
class PCSA(nn.Module):
    def __init__(self, channels):
        super(PCSA, self).__init__()
        # Channel attention captures variations in heart rate frequency bands
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(channels, channels // 4)
        self.fc2 = nn.Linear(channels // 4, channels)

        # Spatial attention: Increased kernel size (31) and added Dilation (4)
        # This expands the receptive field to capture long-range apnea cycles (30-90s)
        self.conv_spatial = nn.Conv1d(channels, channels, kernel_size=31, padding=60, dilation=4)

    def forward(self, x):
        b, c, l = x.size()
        # Channel attention
        y = self.avg_pool(x).view(b, c)
        y = F.relu(self.fc1(y))
        y = torch.sigmoid(self.fc2(y)).view(b, c, 1)
        x = x * y
        # Spatial attention
        s = torch.sigmoid(self.conv_spatial(x))
        x = x * s
        return x

In [8]:
# --- 2. Supporting Network Layers ---
class Conv1DBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size // 2),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class Conv2DBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class TwoConvBranch2D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = Conv2DBlock(channels, channels)
        self.conv2 = Conv2DBlock(channels, channels)
    def forward(self, x): return self.conv2(self.conv1(x))

In [9]:
# --- 3. Final Custom Apnea Model ---
class CustomApneaModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Multiscale transformation with larger kernels for respiratory context
        self.ms1 = Conv1DBlock(1, 32, 3)
        self.ms2 = Conv1DBlock(1, 32, 15)
        self.ms3 = Conv1DBlock(1, 32, 31)

        self.stem_conv1 = Conv2DBlock(1, 32)
        self.pool1 = nn.MaxPool2d(kernel_size=(1, 4), stride=(1, 4))
        self.stem_conv2 = Conv2DBlock(32, 64)
        self.pool2 = nn.MaxPool2d(kernel_size=(1, 4), stride=(1, 4))

        self.branch1 = TwoConvBranch2D(64)
        self.branch2 = TwoConvBranch2D(64)
        self.branch3 = TwoConvBranch2D(64)

        self.fuse = Conv2DBlock(64 * 3, 96)
        self.pcsa = PCSA(96)
        
        self.fc1 = nn.Linear(96, 64)
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        m1 = self.ms1(x)
        m2 = self.ms2(x)
        m3 = self.ms3(x)

        # Max stacking helps highlight sharp R-peak fluctuations (EDR)
        ms = torch.stack([m1.max(dim=1)[0], m2.max(dim=1)[0], m3.max(dim=1)[0]], dim=1).unsqueeze(1)

        z = self.pool1(self.stem_conv1(ms))
        z = self.pool2(self.stem_conv2(z))

        z = torch.cat([self.branch1(z), self.branch2(z), self.branch3(z)], dim=1)
        z = self.fuse(z)

        B, C, H, W = z.shape
        z = self.pcsa(z.view(B, C, H * W))

        # Global aggregation
        z = z.mean(dim=-1)
        z = F.relu(self.fc1(z))
        return self.fc2(z)

In [10]:
# --- 4. Training Initialization ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomApneaModel().to(device)

# Adjusted class weights to prioritize apnea detection sensitivity
# Normal: 23555, Apnea: 14667
class_weights = torch.tensor([0.81, 1.30], dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [11]:
# --- 5. Evaluation Function ---
def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad(): # Optimization for inference
        for signals, labels in loader:
            signals, labels = signals.to(device), labels.to(device)
            outputs = model(signals)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
    
    # Calculate Sensitivity (Recall for Apnea Class) and Specificity
    # cm[1,1] = True Positives, cm[1,0] = False Negatives
    # cm[0,0] = True Negatives, cm[0,1] = False Positives
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    pos_rate = np.mean(all_preds)
    
    return acc, sensitivity, specificity, pos_rate


In [12]:
# --- 6. Training Loop ---
num_epochs = 100
patience = 12
min_delta = 1e-4

best_acc = -1.0
best_state = None
no_improve = 0

print("Starting Training...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for signals, labels in train_loader:
        signals, labels = signals.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(signals)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * signals.size(0)
    
    epoch_loss = running_loss / len(train_loader.dataset)
    
    # Evaluate on validation set
    val_acc, val_sen, val_spec, _ = evaluate(model, val_loader)
    
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {epoch_loss:.4f} | "
          f"Val Acc: {val_acc:.4f} | Sen: {val_sen:.4f} | Spec: {val_spec:.4f}")
    
    # Save the best model and track early stopping
    if val_acc > best_acc + min_delta:
        best_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1

    if no_improve >= patience:
        print(f"Early stopping at epoch {epoch+1} (no improvement for {patience} epochs).")
        break


Starting Training...
Epoch 1/100 | Loss: 0.4082 | Val Acc: 0.7996 | Sen: 0.9564 | Spec: 0.7019
Epoch 2/100 | Loss: 0.2626 | Val Acc: 0.9029 | Sen: 0.9298 | Spec: 0.8862
Epoch 3/100 | Loss: 0.2292 | Val Acc: 0.8909 | Sen: 0.9489 | Spec: 0.8548
Epoch 4/100 | Loss: 0.2137 | Val Acc: 0.9139 | Sen: 0.9202 | Spec: 0.9100
Epoch 5/100 | Loss: 0.2031 | Val Acc: 0.9105 | Sen: 0.9257 | Spec: 0.9011
Epoch 6/100 | Loss: 0.1958 | Val Acc: 0.9181 | Sen: 0.9080 | Spec: 0.9244
Epoch 7/100 | Loss: 0.1881 | Val Acc: 0.9105 | Sen: 0.9332 | Spec: 0.8964
Epoch 8/100 | Loss: 0.1820 | Val Acc: 0.9186 | Sen: 0.9543 | Spec: 0.8964
Epoch 9/100 | Loss: 0.1774 | Val Acc: 0.9171 | Sen: 0.9380 | Spec: 0.9040
Epoch 10/100 | Loss: 0.1731 | Val Acc: 0.9296 | Sen: 0.9059 | Spec: 0.9444
Epoch 11/100 | Loss: 0.1688 | Val Acc: 0.9103 | Sen: 0.9509 | Spec: 0.8849
Epoch 12/100 | Loss: 0.1654 | Val Acc: 0.9273 | Sen: 0.9134 | Spec: 0.9359
Epoch 13/100 | Loss: 0.1605 | Val Acc: 0.9291 | Sen: 0.9223 | Spec: 0.9333
Epoch 14/100 

In [13]:
# --- 7. Final Test Evaluation ---
if best_state:
    model.load_state_dict(best_state)
    torch.save(best_state, 'best_apnea_pcsa_model.pth')
    
    test_acc, test_sen, test_spec, test_pos_rate = evaluate(model, test_loader)
    
    print("\n" + "="*30)
    print("FINAL TEST METRICS")
    print(f"Accuracy   : {test_acc:.4f}")
    print(f"Sensitivity: {test_sen:.4f} (Apnea Detection)")
    print(f"Specificity: {test_spec:.4f} (Normal Detection)")
    print(f"Positive Rate: {test_pos_rate:.4f}")
    print("="*30)


FINAL TEST METRICS
Accuracy   : 0.9406
Sensitivity: 0.9182 (Apnea Detection)
Specificity: 0.9546 (Normal Detection)
Positive Rate: 0.3803
